# YOUTUBE CHATTBOT 


Step 1 - Indexing

In [115]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "AJtDXIazrMo"  # only the ID, not full URL

try:
    ytt_api = YouTubeTranscriptApi()

    transcript_data = ytt_api.fetch(
        video_id,
        languages = ["en"])

    # Convert transcript snippets into one string
    transcript = " ".join(
        snippet.text for snippet in transcript_data
    )

    print(transcript)

except Exception as e:
    print(f"Error: {e}")

You're the light,
you're the night You're the colour of my blood You're the cure, you're the pain You're the only thing I wanna touch Never knew that it could mean
so much, so much You're the fear, I don't care Cause I've never been so high Follow me through the dark Let me take you past the satellites You can see the world
you brought to life, to life So love me like you do,
la-la-love me like you do Love me like you do,
la-la-love me like you do Touch me like you do,
ta-ta-touch me like you do What are you waiting for? Fading in, fading out
on the edge of the paradise Every inch of your skin is
a Holy Grail I've got to find Only you can set my heart
on fire, on fire, yeah I'll let you set the pace Cause I'm not thinking straight My head's spinning around,
I can't see clear no more What are you waiting for? Love me like you do, la-la-love me
like you do (Like you do) Love me like you do,
la-la-love me like you do Touch me like you do,
ta-ta-touch me like you do What are you waiting fo

Step 2 - Text Splitter

In [116]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.create_documents([transcript])

In [117]:
len(chunks)

2

In [118]:
pip install langchain-huggingface


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Step - 3 Embedding Generation & Storing in Vector Store

In [119]:
from dotenv import load_dotenv
load_dotenv()
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8034.32it/s]


In [120]:
vector_store.index_to_docstore_id


{0: '93183960-cc7b-4853-b033-36161c3bea48',
 1: '07fb02dd-e428-4045-9e85-76982834e1fd'}

In [121]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

Retrival is an Runnable 


In [122]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x380cbd590>, search_kwargs={'k': 3})

Augumentation - Prompt Generate

LLM Generation


In [123]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

Prompt creation


In [124]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are a helpful AI assistant.

Answer the user's question using ONLY the provided context.

If relevent answer are not present say "I dont know"

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
) 

In [125]:
question = "is the topic of aliens discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [126]:
len(retrieved_docs)

2

Since k= 3 , we will get 3 documents over here, so here we can't send all the 3 document as context. Therefore, we will merge the page_content of all the 3 doc into string then sent it as context

#### Concatenation

In [127]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [128]:
print(context_text)

I can't see clear no more What are you waiting for? Love me like you do, la-la-love me
like you do (Like you do) Love me like you do,
la-la-love me like you do Touch me like you do,
ta-ta-touch me like you do What are you waiting for? Love me like you do, la-la-love me
like you do (Like you do) Love me like you do, la-la-love me
like you do (Yeah) Touch me like you do, ta-ta-touch
me like you do (Ooh ooh ooh) What are you waiting for? I'll let you set the pace Cause I'm not thinking straight My head's spinning around,
I can't see clear no more What are you waiting for? Love me like you do, la-la-love me
like you do (Like you do) Love me like you do, la-la-love
me like you do (Yeah) Touch me like you do, ta-ta-touch
me like you do (Ooh ooh ooh) What are you waiting for? Love me like you do, la-la-love me
like you do  (Like you do) Love me like you do, la-la-love me
like you do (Whoa) Touch me like you do,
ta-ta-touch me like you do What are you waiting for?

You're the light,
you're the

Final Prompt 

In [129]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [130]:
print(final_prompt)

text='\nYou are a helpful AI assistant.\n\nAnswer the user\'s question using ONLY the provided context.\n\nIf relevent answer are not present say "I dont know"\n\nContext:\nI can\'t see clear no more What are you waiting for? Love me like you do, la-la-love me\nlike you do (Like you do) Love me like you do,\nla-la-love me like you do Touch me like you do,\nta-ta-touch me like you do What are you waiting for? Love me like you do, la-la-love me\nlike you do (Like you do) Love me like you do, la-la-love me\nlike you do (Yeah) Touch me like you do, ta-ta-touch\nme like you do (Ooh ooh ooh) What are you waiting for? I\'ll let you set the pace Cause I\'m not thinking straight My head\'s spinning around,\nI can\'t see clear no more What are you waiting for? Love me like you do, la-la-love me\nlike you do (Like you do) Love me like you do, la-la-love\nme like you do (Yeah) Touch me like you do, ta-ta-touch\nme like you do (Ooh ooh ooh) What are you waiting for? Love me like you do, la-la-love 

Step 4 - Generation


In [131]:
answer = llm.invoke(final_prompt)
print(answer.content)

I dont know


### Building a chain in order to automate

In [132]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [133]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text
    

In [134]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [135]:
parser = StrOutputParser()

In [136]:
main_chain = parallel_chain | prompt | llm | parser

In [137]:
main_chain.invoke("Can you summarize the video")

'I dont know'